# 2.1.3 Improved LSTM: training catalogue export

**Owner:** Winston Ho Rui Wei  
**Status:** prepared for Colab; no training predictions or metrics are assumed.

This companion notebook reloads the selected **LSTM + freshness** run
`20261104_175920_improved_seed42` and generates top-200 recommendations for
all **232,887 training requests**, using the shared **20,738-article SMALL
catalogue** and the existing publication-time eligibility policy.

Weights, preprocessing and the selected freshness coefficient remain fixed.
There is no model fitting, tuning or business reranking. Training metrics are
**in-sample diagnostics**, because these requests contributed to model fitting;
they are not out-of-fold predictions or independent evidence of generalisation.

## 1. Colab setup

Choose **A100 GPU** and **High RAM**, then **Run all**. No helper ZIP, API key or
GitHub clone is needed: the original inference helpers load from the saved model
bundle. Keep the live shared-folder shortcut under `MyDrive/Colab Notebooks`
or directly under `MyDrive`. Authorise Drive when prompted.

Files are staged on the Colab runtime disk, checked, and copied into the
**existing improved-run folder**. Validation/test recommendation files and
model files are not publication targets.


In [ ]:
from pathlib import Path
from dataclasses import fields
import hashlib, importlib.util, json, shutil, sys, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from google.colab import drive

drive.mount('/content/drive')
ROOT_OPTIONS = [
    Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder'),
    Path('/content/drive/MyDrive/BT4222 Shared Final Folder'),
]
DATA_ROOT = next(path for path in ROOT_OPTIONS if path.is_dir())
SOURCE_RUN_ID = '20261104_175920_improved_seed42'
SOURCE_RUN = DATA_ROOT / 'Experiments/2.1_lstm/2.1.3_improved_run' / SOURCE_RUN_ID
BUNDLE = SOURCE_RUN / 'model'
WORK = Path('/content/lstm_training_export') / SOURCE_RUN_ID
WORK.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), 'Select a GPU runtime before running this notebook.'
DEVICE = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
EXPORT_K, METRIC_KS = 200, (5, 100, 200)
REQUEST_BATCH, PAIR_CHUNK = 16, 131072

def digest(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def log(*parts):
    print(*parts, flush=True)

log('GPU:', torch.cuda.get_device_name())
log('Saved model and upload destination:', SOURCE_RUN)
log('Local staging folder:', WORK)


## 2. Reload the selected model

Use the saved model's own code, fitted transforms and frozen article lookup.
The original lookup contains 125,541 articles plus its padding row; it is kept
intact so stored row mappings remain valid. The **candidate index below is
restricted to the shared 20,738 original article IDs**. Lookup size does not
define the recommendation pool.

Helper code is copied to the runtime before importing, so Python does not write
cache files into the submitted Drive folder.


In [ ]:
CODE = WORK / 'saved_model_code'
shutil.copytree(BUNDLE / 'code', CODE, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__'))

def load_saved_module(name):
    spec = importlib.util.spec_from_file_location(name, CODE / f'{name}.py')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

D = load_saved_module('lstm_data')
M = load_saved_module('lstm_model')
F = load_saved_module('lstm_inference')
PREPROCESSING = json.loads((BUNDLE / 'preprocessing.json').read_text())
SCORE_RECIPE = json.loads((BUNDLE / 'score_recipe.json').read_text())
FRESHNESS_WEIGHT = float(SCORE_RECIPE['freshness_weight'])
SOURCE_MODEL_SHA256 = digest(BUNDLE / 'model_state.pt')
cfg = M.ModelConfig.from_json(json.loads((BUNDLE / 'model_config.json').read_text()))
with np.load(BUNDLE / 'article_inputs.npz', allow_pickle=False) as saved:
    article_fields = {field.name: saved[field.name] for field in fields(D.ArticleTable)}
article_fields['numeric_names'] = article_fields['numeric_names'].tolist()
ARTICLES = D.ArticleTable(**article_fields)
model = M.build_model(cfg, M.article_table_tensors(ARTICLES, DEVICE), DEVICE)
model.load_state_dict(torch.load(BUNDLE / 'model_state.pt', map_location=DEVICE, weights_only=True))
model.eval().requires_grad_(False)
log('Loaded selected weights:', SOURCE_RUN_ID, '| fixed freshness coefficient:', FRESHNESS_WEIGHT)


## 3. Prepare training requests and the SMALL candidate pool

Read only the training feature/history inputs and the shared article snapshots.
Reuse the original chronological latest-50 history rule and saved numeric/device
transforms. No new histories are inferred from labels, and no vocabulary or
scaler is fitted. Click labels are used only to measure recovery after scoring.

Check the pool fingerprint against the refreshed validation/test declaration.
Missing-vector candidates remain eligible. Future publications are excluded;
known clicks excluded by eligibility stay in metric denominators.


In [ ]:
PATHS, _ = D.resolve_paths(DATA_ROOT, required=[
    'features_train', 'history_train', 'articles_train_val', 'articles_test'])
frame, info = D.load_candidate_table(PATHS['features_train'], 'train')
offsets = info.pop('offsets')
history = D.build_history(PATHS['history_train'], pd.Timestamp(frame.impression_time.min()),
    PREPROCESSING['history_max_len'], users_needed=frame.user_id.to_numpy())
a = D.build_split_arrays('train', frame, offsets, history, ARTICLES,
    PREPROCESSING['numeric_stats'], PREPROCESSING['device_columns'])
TENSORS = M.SplitTensors(a, DEVICE)
del frame, history

CATALOGUE, _ = D.load_catalogue([PATHS['articles_train_val'], PATHS['articles_test']])
SMALL_IDS = np.sort(CATALOGUE.index.to_numpy(dtype=np.int64))
assert len(SMALL_IDS) == 20738, f'Expected SMALL catalogue; found {len(SMALL_IDS):,} articles.'
assert a.n_impressions == 232887, f'Expected all training requests; found {a.n_impressions:,}.'
POOL = json.loads((SOURCE_RUN / 'evaluation/catalogue.json').read_text())
pool_hash = hashlib.sha256(SMALL_IDS.astype('<i8').tobytes()).hexdigest()
assert pool_hash == POOL['article_ids_sha256'], 'Candidate IDs differ from the refreshed evaluation pool.'
POOL['policy']['same_pool_for_splits'] = ['train', 'validation', 'test']
INDEX = D.build_catalogue_index('train', 'SMALL', SMALL_IDS, ARTICLES)
current_publication = pd.to_datetime(CATALOGUE.loc[INDEX.article_ids, 'published_time'])
assert np.array_equal(current_publication.to_numpy(dtype='datetime64[us]').astype('int64'), INDEX.published_us)
ELIGIBILITY = D.eligibility_summary(a, INDEX)
CLICKS = D.clicked_eligibility(a, INDEX, ARTICLES)
BIDS = D.behaviour_ids(a.user_id, a.impression_id)
log('Training requests:', a.n_impressions, '| recorded candidate pairs:', a.n_candidates)
log('Candidate catalogue:', len(SMALL_IDS), '| eligible pairs to score:', ELIGIBILITY['eligible_pairs'])
log('Known-click eligibility:', CLICKS['summary'])


## 4. Export genuine top-200 rankings

Use the existing batched float32 scorer and columnar Parquet writer. It scores
every eligible candidate, caches history/article encodings, and retains the
best 200. Ties use ascending original article ID. No clicked or originally shown
articles are inserted into the shortlist, and no exhaustive score matrix is saved.

The fixed final score is `neural_logit - freshness_weight * log1p(age_hours)`.
It is an uncalibrated ranking value. Report macro known-click Recall and
observed-click nDCG at 5, 100 and 200 as **training in-sample metrics**. Unexposed
articles have unknown outcomes; they are not new negative training labels.


In [ ]:
scorer = F.BatchedCatalogueScorer(model, TENSORS, INDEX,
    PREPROCESSING['numeric_stats']['log_candidate_age_h'], PAIR_CHUNK, REQUEST_BATCH)
lookup = D.clicks_by_request(CLICKS)
hits = {}
writer = F.ColumnarShortlistWriter(WORK, 'train', EXPORT_K)
started, next_progress = time.time(), 10000
try:
    for block, counts, outputs in scorer.batches(np.arange(a.n_impressions), EXPORT_K, (FRESHNESS_WEIGHT,)):
        selected = outputs[FRESHNESS_WEIGHT]
        writer.add_batch(BIDS[block], a.imp_time_us[block], counts,
                         selected['article_ids'], selected['scores'])
        for row, i in enumerate(block):
            size = min(EXPORT_K, int(counts[row]))
            result = D.click_hits(lookup, int(i), selected['article_ids'][row, :size], METRIC_KS)
            if result is not None:
                hits[int(i)] = result
        done = int(block[-1]) + 1
        if done >= next_progress:
            elapsed = time.time() - started
            log('train:', f'{done:,}/{a.n_impressions:,}', '| elapsed min:', round(elapsed/60, 1),
                '| ETA min:', round(elapsed/done*(a.n_impressions-done)/60, 1))
            next_progress += 10000
finally:
    files = writer.close()

LOCAL_CHECK = D.validate_catalogue_exports(a, INDEX, files, EXPORT_K)
TRAIN_METRICS = D.recall_from_hits(hits, CLICKS, METRIC_KS, requests_scored=a.n_impressions)
RECORD = {
    'scope': 'all training requests; in-sample descriptive metrics; no model selection',
    'requests': a.n_impressions, 'eligible_pairs': ELIGIBILITY['eligible_pairs'],
    'files': {name: str(Path(files[name]).relative_to(WORK)) for name in ('rankings','scores','manifest')},
    'coverage': ELIGIBILITY, 'metrics': TRAIN_METRICS, 'freshness_weight': FRESHNESS_WEIGHT,
    'seconds': time.time()-started, 'integrity_check': LOCAL_CHECK,
}
D.write_json(RECORD, WORK / 'catalogue/train.json')
D.write_json(POOL, WORK / 'evaluation/train_catalogue.json')
SUMMARY = {
    'scope': 'training in-sample diagnostic; not out-of-fold or final holdout performance',
    'source_run_id': SOURCE_RUN_ID, 'source_model_sha256': SOURCE_MODEL_SHA256,
    'catalogue_articles': len(SMALL_IDS), 'catalogue_id_fingerprint_sha256': pool_hash,
    'requests': a.n_impressions, 'score_recipe': SCORE_RECIPE, 'metrics': TRAIN_METRICS,
    'model_training_performed': False, 'tuning_performed': False,
    'local_export_check': LOCAL_CHECK, 'export_seconds': RECORD['seconds'],
}
log('Local training export complete:', LOCAL_CHECK)
display(pd.DataFrame([{'split':'train', 'scope':'in-sample', **{
    key:value for key,value in TRAIN_METRICS.items()
    if key.startswith(('known_click_recall@','observed_click_ndcg@'))}}]))
del scorer, lookup, hits
torch.cuda.empty_cache()


## 5. Persist to the existing run and verify

Copy only the new training outputs into the existing improved-run folder.
Read the persistent Parquet files back to verify complete request coverage,
eligibility counts and a score/ranking probe. Then update the export declarations,
training metrics and file-size manifest to show the restored **20k-pool training
exports**. Existing validation/test metrics and model-selection evidence remain
in their original sections. Original model lookup files remain intact.

The earlier cleanup removed obsolete 125k training recommendations. These new
files replace that missing training handoff with the shared SMALL pool. The run's
`submission_artifacts.json` records the updated scope.


In [ ]:
def stage_training_metadata(source_run, work, record, pool, summary, data_paths):
    """Restore train declarations while preserving evaluation/model provenance."""
    source_run, work = Path(source_run), Path(work)
    def read(name):
        return json.loads((source_run / name).read_text())
    def save(name, value):
        target = work / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(value, indent=2) + '\n')

    config = read('run_config.json')
    exports = read('export_manifest.json')
    metrics = read('metrics.json')
    status = read('submission_artifacts.json')
    for document in (config, exports):
        document.setdefault('catalogue_by_split', {})['train'] = pool
        document.setdefault('evaluation_scope', {}).update(
            training_exports_updated=True, training_exports_retained=True)
    config.setdefault('catalogue_exports', {})['train'] = record
    config['export_request_scope'] = 'all train, validation and project holdout requests over the shared SMALL catalogue'
    config['train_predictions'] = 'Training in-sample diagnostics; not out-of-fold; fixed selected weights and freshness.'
    exports.setdefault('splits', {})['train'] = record
    metrics['training_catalogue'] = summary
    status['current_export_scope'] = 'Complete train, validation and holdout recommendations over the shared SMALL catalogue; train is in-sample.'
    status['training_export'] = 'evaluation/training_export.json'
    status['removed_files'] = [path for path in status.get('removed_files', []) if path not in data_paths]
    status['results'] = 'Training recommendations regenerated over SMALL with fixed weights/freshness; validation/test predictions and original model files unchanged.'
    for name, value in [('run_config.json',config), ('export_manifest.json',exports),
                        ('metrics.json',metrics), ('submission_artifacts.json',status)]:
        save(name, value)
    payload = list(data_paths) + ['run_config.json','export_manifest.json','metrics.json','submission_artifacts.json']
    manifest = read('manifest.json')
    for name in payload:
        manifest[name] = (work / name).stat().st_size
    save('manifest.json', manifest)
    return payload + ['manifest.json']


In [ ]:
DATA_PATHS = [
    RECORD['files']['rankings'], RECORD['files']['scores'], RECORD['files']['manifest'],
    'catalogue/train.json', 'evaluation/train_catalogue.json',
]
for relative in DATA_PATHS:
    D._copy_atomic(WORK / relative, SOURCE_RUN / relative)

persistent_files = {name: str(SOURCE_RUN / relative) for name,relative in RECORD['files'].items()}
PERSISTENT_CHECK = D.validate_catalogue_exports(a, INDEX, persistent_files, EXPORT_K)
assert digest(BUNDLE / 'model_state.pt') == SOURCE_MODEL_SHA256
SUMMARY['persistent_export_check'] = PERSISTENT_CHECK
SUMMARY['source_weights_unchanged'] = True
SUMMARY['completed_at'] = pd.Timestamp.now(tz='Asia/Singapore').isoformat()
D.write_json(SUMMARY, WORK / 'evaluation/training_export.json')
ALL_DATA_PATHS = DATA_PATHS + ['evaluation/training_export.json']
payload = stage_training_metadata(SOURCE_RUN, WORK, RECORD, POOL, SUMMARY, ALL_DATA_PATHS)
for relative in payload:
    if relative not in DATA_PATHS:
        D._copy_atomic(WORK / relative, SOURCE_RUN / relative)

saved_summary = json.loads((SOURCE_RUN / 'evaluation/training_export.json').read_text())
assert saved_summary == json.loads((WORK / 'evaluation/training_export.json').read_text())
log('Training export persisted and verified:', True)
log('Training top 200:', SOURCE_RUN / RECORD['files']['rankings'])
log('Retained training scores:', SOURCE_RUN / RECORD['files']['scores'])
log('Training metrics:', SOURCE_RUN / 'evaluation/training_export.json')
log('Requests:', PERSISTENT_CHECK['requests'], '| score rows:', PERSISTENT_CHECK['score_rows'],
    '| candidate catalogue:', len(SMALL_IDS))


## 6. Training-metric handoff

The persistent run now contains:

| File | Purpose |
| --- | --- |
| `rankings/train/submission_top_200.parquet` | One native-list ranking per training request; original composite behaviour IDs |
| `predictions/train/submission_article_scores.parquet` | Scores and ranks for each retained article |
| `predictions/train/eligibility_manifest.parquet` | Every request's timestamp, eligible count and shortlist length |
| `catalogue/train.json` | Training export counts, metrics and timing |
| `evaluation/train_catalogue.json` | The same SMALL pool declaration used by validation/test |
| `evaluation/training_export.json` | In-sample metrics, fixed-score provenance and persistent-copy checks |

`run_config.json`, `export_manifest.json`, `metrics.json`,
`submission_artifacts.json` and `manifest.json` include the new training export.
The complete original recorded-slate training diagnostics remain available for
separate AUC analysis; catalogue top-200 rankings alone cannot establish AUC.

Wait for **`Training export persisted and verified: True`**, then download this
executed notebook. Its actual export results can be incorporated into the
original improved-run notebook while retaining the original training outputs.

Report training results as descriptive fit diagnostics alongside validation and
holdout results on the same catalogue. They do not establish generalisation and
must not be used as out-of-fold inputs for a learned stacking model. This notebook
does not generate training RRF or business-reranked results.
